# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [14]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [13]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'profile page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'About Us page', 'url': 'https://edwarddonner.com/'},
  {'type': 'LinkedIn Company profile page',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [15]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
baidu/Unlimited-OCR
Updated
5 days ago
•
758k
•
1.64k
empero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF
Updated
4 days ago
•
1.25M
•
1.23k
zai-org/GLM-5.2
Updated
about 10 hours ago
•
176k
•
3.24k
deepreinforce-ai/Ornith-1.0-35B-GGUF
Updated
7 days ago
•
285k
•
644
deepreinforce-ai/Ornith-1.0-9B-GGUF
Updated
7 days ago
•
255k
•
392
Browse 2M+ models
Spaces
Running
on
Zero
Agents
320
Pro Realism

In [17]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbaidu/Unlimited-OCR\nUpdated\n5 days ago\n•\n758k\n•\n1.64k\nempero-ai/Qwythos-9B-Claude-Mythos-5-1M-GGUF\nUpdated\n4 days ago

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a vibrant AI community building the future of machine learning. It serves as the premier collaboration platform where researchers, developers, and enterprises can create, discover, and share machine learning models, datasets, and applications. With over 2 million models and half a million datasets available, Hugging Face fosters a dynamic environment that drives innovation and AI accessibility worldwide.

---

## What We Offer

- **Models Hub:** Access and contribute to 2M+ models spanning various ML tasks like NLP, computer vision, and more.
- **Datasets:** Explore 500k+ datasets openly available for research and development.
- **Spaces:** Run and deploy AI applications easily on user-friendly interfaces.
- **Buckets & Storage:** Secure cloud storage for data and model hosting.
- **Enterprise Solutions:** Customized support, inference endpoints, and enterprise-grade tools via Hugging Face PRO.
- **Community & Collaboration:** Active forums, Discord channels, GitHub repositories, and blogs facilitate continuous learning and sharing.

---

## Company Culture

Hugging Face thrives on openness, collaboration, and innovation. It is a community-first organization committed to building tools that empower researchers and practitioners alike with transparency and inclusivity. The platform encourages sharing and co-creation, creating a unique ecosystem where ideas flourish and AI development is democratized.

---

## Customers & Community

- **Researchers & Scientists:** Access cutting-edge models and datasets for academic and applied research.
- **Developers & ML Engineers:** Build, deploy, and share AI applications with ease using Spaces and API services.
- **Enterprises:** Leverage Hugging Face PRO for scalable AI infrastructure, model hosting, and dedicated support.
- **Open Source Contributors:** Participate in a global community actively advancing AI through code, papers, and discussions.

Hugging Face powers AI innovation across industries including healthcare, finance, education, and creative arts.

---

## Careers & Opportunities

Hugging Face invites passionate individuals to join its mission of advancing AI for the benefit of all. Careers at Hugging Face offer roles in engineering, research, product management, community engagement, and enterprise solutions. Working here means being part of a cutting-edge, mission-driven company with a global impact.

---

## Connect With Us

- **Website:** [huggingface.co](https://huggingface.co/)
- **Community Forums & Discord:** Engage with our active AI community
- **GitHub:** Contribute or explore open source projects
- **Blog & Daily Papers:** Stay updated with the latest breakthroughs and company news

---

## Hugging Face — The Home of Machine Learning Collaboration

Join the movement to build the future of AI together.

---

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face  
**Hugging Face** is the AI community building the future of machine learning. It serves as a collaborative platform where data scientists, researchers, developers, and enterprises come together to create, discover, and share machine learning models, datasets, and applications. Hugging Face is recognized globally as *The Home of Machine Learning*, empowering thousands to build and deploy cutting-edge AI technologies.

---

## What Hugging Face Offers

- **Models:** Access and contribute to a collection of over 2 million machine learning models spanning various AI tasks and domains including natural language processing, computer vision, and more. Models are constantly updated by an active community.
  
- **Datasets:** Explore and collaborate on more than 500,000 datasets available for machine learning projects, covering diverse fields and languages.

- **Spaces:** Host and showcase machine learning apps with an easy interface, enabling real-time AI app deployments and user interactions.

- **Buckets & Storage:** Enterprise-grade storage solutions to manage data and models securely and efficiently.

- **Enterprise Solutions:** Tailored support and services including inference providers, endpoints, and Hugging Face PRO for companies scaling ML production.

---

## Community & Collaboration  
Hugging Face thrives on its vibrant and inclusive community platform where:

- Developers and researchers **collaborate openly** on shared projects.
- Members participate in forums, Discord channels, and GitHub repositories to exchange knowledge.
- Learning resources such as blogs, daily paper reviews, and tutorials foster continual growth.
- Over 1 million applications and AI-powered tools are discoverable and reusable.

---

## Company Culture  
Hugging Face embodies a culture of openness, innovation, and community focus. It prioritizes:

- **Transparency:** Open collaboration on models and datasets.
- **Inclusivity:** Welcoming diverse talents across disciplines and experiences.
- **Innovation:** Pushing the boundaries of AI through shared knowledge and technology.
- **Empowerment:** Providing tools and resources that accelerate AI development worldwide.

---

## Customers & Users  
Hugging Face serves a wide range of users:

- **AI Researchers & Academics:** Who require access to state-of-the-art models and datasets for experimentation.
- **Developers & Data Scientists:** Building and deploying ML-powered applications efficiently.
- **Enterprises:** Seeking robust AI infrastructure and custom support for large-scale ML deployment.
- **AI Enthusiasts & Students:** Looking for a learning ground and community engagement.

---

## Career Opportunities  
Join Hugging Face and be part of shaping the future of AI! The company is continuously seeking passionate individuals in fields such as:

- Machine Learning & Research
- Software Engineering & DevOps
- Community & Developer Relations
- Product Management & Design
- Enterprise Solutions and Customer Support

Hugging Face encourages creativity, ownership, and growth among its team members. Opportunities open globally, embracing remote and flexible working models.

---

## Quick Facts

- **2 Million+ Models**
- **500,000+ Datasets**
- **1 Million+ AI Applications**
- Active Community across Discord, GitHub, and Forums
- Enterprise-grade AI support services

---

## Connect & Learn More  
- Visit: [huggingface.co](https://huggingface.co)  
- Join the Community: Discord, Forum, GitHub  
- Explore Documentation and Tutorials  
- Follow the Blog for Latest AI Insights

---

**Hugging Face** — Empowering the AI community to build the future, together.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>